In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

## RAG 절차
- 1. docx문서를 읽는다(python-docx) : \n은 제외
- 2. 읽어온 문서를 쪼갠다(tiktoken이용)
    * 모델의 context window와 비용 확인
    * chunk(문서를 쪼갠 하나)가 길면 비용과 시간이 많이 소요
- 3. 쪼갠 문서를 임베딩 -> vector Store에 저장 => chroma(local vector Database) / pinecorn(web cloude vector Database)
- 4. 질문과 vector DB의 유사도 검색
- 5. 유사도검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성
# 1. 문서를 읽는다(python-docx)
- pip install -U -q lanchain-community python-docx docx2txt 

In [3]:
%pip install -U -q langchain-community python-docx docx2txt 

Note: you may need to restart the kernel to use updated packages.


In [5]:
# 이렇게 docx파일을 읽어오면 한꺼번에 가져옴(\n포함)
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
document = loader.load() #\n을 포함하면 한꺼번에 load
len(document)

1

In [16]:
# 문단씩 가져오기
from docx import Document
documents=Document('./tax_docx/소득세법(법률)(제21221호)(20260701).docx')
documents.paragraphs

 ...]

In [19]:
len(documents.paragraphs) #\n마다 새로운 paragraghs로 생성
for paragraphs in documents.paragraphs[:5]:
    #print(dir(paragraphs))
    print('------------------------')
    print(paragraphs.text)

------------------------

------------------------
소득세법
------------------------
[시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정]
------------------------
재정경제부(재산세제과(양도소득세)) 044-215-4312
------------------------
재정경제부(소득세제과(근로소득)) 044-215-4216


In [ ]:
# \n이 제거된 문서 전체 내용을 full_text

In [24]:
full_text=''
for paragraph in documents.paragraphs:
    full_text+=paragraph.text +' '
full_text[:100]

' 소득세법 [시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정] 재정경제부(재산세제과(양도소득세)) 044-215-4312 재정경제부(소득세제'

# 2. 문서를 쪼갠다
- pip install tiktoken : openAI의 공식 토크나이저 라이브러리
    - full_text -> 토큰단위로 쪼개서 숫자
    - 숫자 -> 문자 

In [25]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [29]:
import tiktoken
encoder =tiktoken.encoding_for_model('gpt-4o-mini')
encoding=encoder.encode(full_text)
#숫자리스트 ->문자들
decoded = encoder.decode(encoding)

In [32]:
decoded[:20]

' 소득세법 [시행 2026. 7. 1'

In [33]:
print(f'글자수:{len(full_text)}')
print(f'토근수:{len(encoding)}')
print('문자들:',decoded[:10])
print('숫자들:',encoding[:10])

글자수:248301
토근수:162419
문자들:  소득세법 [시행 
숫자들: [22183, 64328, 11734, 22070, 723, 5637, 15719, 220, 1323, 21]


In [34]:
print(encoder.encode('소득세법 어쩌구 저쩌구'))

[11226, 64328, 11734, 22070, 21252, 152176, 234, 10997, 30832, 152176, 234, 10997]


In [38]:
print(encoder.encode('Hello'),encoder.encode('안녕'),encoder.encode('우리'))

[13225] [14307, 98931] [144799]


In [39]:
print(list(range(0,162419,1500)))

[0, 1500, 3000, 4500, 6000, 7500, 9000, 10500, 12000, 13500, 15000, 16500, 18000, 19500, 21000, 22500, 24000, 25500, 27000, 28500, 30000, 31500, 33000, 34500, 36000, 37500, 39000, 40500, 42000, 43500, 45000, 46500, 48000, 49500, 51000, 52500, 54000, 55500, 57000, 58500, 60000, 61500, 63000, 64500, 66000, 67500, 69000, 70500, 72000, 73500, 75000, 76500, 78000, 79500, 81000, 82500, 84000, 85500, 87000, 88500, 90000, 91500, 93000, 94500, 96000, 97500, 99000, 100500, 102000, 103500, 105000, 106500, 108000, 109500, 111000, 112500, 114000, 115500, 117000, 118500, 120000, 121500, 123000, 124500, 126000, 127500, 129000, 130500, 132000, 133500, 135000, 136500, 138000, 139500, 141000, 142500, 144000, 145500, 147000, 148500, 150000, 151500, 153000, 154500, 156000, 157500, 159000, 160500, 162000]


In [51]:
import tiktoken
def split_text(full_text=full_text,chunk_size=1500):
    'full_text를 chunk_size 토큰만큼 쪼개서 chunk_list로 반환'
    encoder=tiktoken.encoding_for_model('gpt-4o-mini')  #gpt nano모델은 안됨
    total_encoding = encoder.encode(full_text) # full_text를 토큰단위로 숫자 list
    total_token_count = len(total_encoding) #full_text의 토큰 수 
    chunk_list=[]
    for i in range(0,total_token_count,chunk_size):
        chunk_encoding=total_encoding[i:i+chunk_size]
        chunk_decoded=encoder.decode(chunk_encoding)
        #print(chunk_encoding,'=>',chunk_decoded)
        chunk_list.append(chunk_decoded)
    return chunk_list
    

In [52]:
example_text ="소득세법 법률 일부 개정함 이자소득 배당소득 무시기 무시기 무기식"
split_text(full_text=example_text,chunk_size=10)

['소득세법 법률 일부 개정함', ' 이자소득 배당소득 무시', '기 무시기 무기식']

In [54]:
chunk_list =split_text(full_text,1500)
len(chunk_list)

109

In [55]:
chunk_list[0]

' 소득세법 [시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정] 재정경제부(재산세제과(양도소득세)) 044-215-4312 재정경제부(소득세제과(근로소득)) 044-215-4216 재정경제부(금융세제과(이자소득, 배당소득)) 044-215-4233 재정경제부(소득세제과(사업소득, 기타소득)) 044-215-4217 재정경제부(국제조세제도과(비거주자)) 044-215-4651  제1장 총칙 <개정 2009. 12. 31.>  제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다. [본조신설 2009. 12. 31.] [종전 제1조는 제2조로 이동 <2009. 12. 31.>]  제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.> 1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다. 2. “비거주자”란 거주자가 아닌 개인을 말한다. 3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다. 4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다. 5. “사업자”란 사업소득이 있는 거주자를 말한다. ② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다. [본조신설 2009. 12. 31.]  제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다. 1. 거주자 2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인 ② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수한 소득세를 납부할 의무를 진다. 1. 거주자 2. 비거주자 3. 내국법인 4. 외국법인의 국내지점 또는 국내영업소(출장소, 그 밖에 이에 준하는 것을 포함한다

 # 3. 쪼갠 문서를 임베딩 -> vector Store(인메모리)에 저장 
 - pip install chromadb

In [57]:
%pip install -q chromadb --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


In [68]:
import chromadb
chroma_client=chromadb.Client()

In [69]:
#collection은 RDB의 테이블 개념
collection_name ='tax_collection'

In [70]:
#임베딩 객체 
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
load_dotenv()
import os
openai_key = os.getenv('OPENAI_API_KEY')
openai_embedding = OpenAIEmbeddingFunction(
    api_key=openai_key,
    model_name='text-embedding-3-large'
)

In [67]:
# collection에 입력하 때 사용될 id들(문자)
ids=[str(id) for id in range(len(chunk_list))]
print(ids)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108']


In [71]:
#chromaDB에 (인메모리 vector Store==vector Database)에 collection 생성
tax_collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=openai_embedding
        )

In [72]:
tax_collection.add(documents=chunk_list,ids=ids)

In [83]:
#tax_collection확인 절차
results = tax_collection.get(include=['embeddings','documents','metadatas'])
print('전체 chunk수:',len(results['ids']))
print('ID목록(처음5개까지만):', results['ids'][:5])
print('임베딩목록(처음5개까지만):', results['embeddings'][:5])
print('documents목록(처음5개까지만):', results['documents'][:5])

전체 chunk수: 109
ID목록(처음5개까지만): ['0', '1', '2', '3', '4']
임베딩목록(처음5개까지만): [[ 0.02467346 -0.02632141  0.00182438 ... -0.00331497 -0.0137558
  -0.01914978]
 [ 0.03497314 -0.01469421 -0.00390434 ... -0.00169182 -0.02319336
  -0.01204681]
 [ 0.0557251   0.00418091 -0.00763702 ... -0.00099087 -0.00484085
  -0.01533508]
 [ 0.03567505 -0.03616333 -0.00316048 ... -0.00646973 -0.02113342
  -0.01062012]
 [ 0.0125885  -0.026474   -0.00330925 ... -0.00468445 -0.01100922
  -0.01321411]]
documents목록(처음5개까지만): [' 소득세법 [시행 2026. 7. 1.] [법률 제21221호, 2025. 12. 23., 일부개정] 재정경제부(재산세제과(양도소득세)) 044-215-4312 재정경제부(소득세제과(근로소득)) 044-215-4216 재정경제부(금융세제과(이자소득, 배당소득)) 044-215-4233 재정경제부(소득세제과(사업소득, 기타소득)) 044-215-4217 재정경제부(국제조세제도과(비거주자)) 044-215-4651  제1장 총칙 <개정 2009. 12. 31.>  제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다. [본조신설 2009. 12. 31.] [종전 제1조는 제2조로 이동 <2009. 12. 31.>]  제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 

# 4. 질문과 vector DB의 유사도 검색

In [84]:
query ="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_doc=tax_collection.query(query_texts=query,
                                  n_results=3 # 유사도가 높은 상위 3개 추출(기본값은10)
                                  
                                  )

In [85]:
#q1 query를 임베딩은??? 
# query를 임베딩 숫자변환    // data를 임베딩 숫자변환 -- > 유사도 비교?
#q2 RAG는 LLM이 필수로 붙어야하는지?

retrieved_doc['documents']

{'ids': [['3', '4', '5']],
 'embeddings': None,
 'documents': [[' 제12조(비과세소득) 다음 각 호의 소득에 대해서는 소득세를 과세하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31., 2025. 10. 1., 2025. 12. 23.> 1. 「공익신탁법」에 따른 공익신탁의 이익 2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득 가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득 나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다. 다. 대통령령으로 정하는 농어가부업소득 라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득 마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 3천만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다. 바. 대통령령으로 정하는 작물재배업에서 발생하는 소득 사. 대통령령으로 정하는 어로어업 또는 양식어업에서 

# 5. 유사도 검색으로 나온 문서를 LLM에 질문과 같이 전달

In [87]:
from openai import OpenAI
load_dotenv()
client=OpenAI()
response=client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[
        {'role':'system',
         'content':f'''당신은 한국 소득세 전문가 입니다. 아래의 내용을 참고해서 사용자 질문에 답변해주세요{retrieved_doc['documents'][0]}'''
        },
        {'role':'user','content':query}
    ]
)

In [89]:
print(response.choices[0].message.content)

연봉 5천만원인 직장인의 소득세는 여러 요소에 따라 달라질 수 있지만, 기본적인 계산 방법을 통해 대략적인 소득세를 계산해드릴 수 있습니다.

1. **종합소득금액**: 연봉 5천만원
2. **소득공제**: 기본공제, 인적공제, 보험료공제 등 적정한 공제를 적용해야 합니다.
   - 기본공제: 본인 150만원
   - 인적공제: 배우자와 자녀 등 추가 공제 대상
   - 연금보험료, 건강보험료, 기부금 등 다른 공제도 고려 가능
3. **과세표준**: 종합소득에서 공제를 뺀 금액
4. **세율 적용**: 과세표준에 따라 세율이 결정됩니다.

**2023년도 소득세율(누진세율 적용)**:
- 1,200만원 이하: 6%
- 1,200만원 초과 4,600만원 이하: 15%
- 4,600만원 초과 8,800만원 이하: 24%

예를 들어, 기본공제로 본인만 공제한다고 가정하고:
- 연봉 5천만원에서 기본공제 150만원을 뺀 4,850만원을 기준으로 하여 세율을 적용해 보겠습니다.

1. 1,200만원에 대한 세금: 1,200만원 × 6% = 72만원
2. 1,200만원 초과 4,600만원 이하(3,400만원)에 대한 세금: 3,400만원 × 15% = 510만원
3. 4,600만원 초과 4,850만원 이하(250만원)에 대한 세금: 250만원 × 24% = 60만원

총 소득세는 72 + 510 + 60 = 642만원이 됩니다.

다만, 최종적으로 납부해야 할 세액은 여기서 다시 세액공제나 감면 등을 고려하여 조정될 수 있습니다. 더 구체적인 계산을 원하시면, 각종 공제 항목과 본인의 상황을 고려하여 세무사와 상담하시기를 추천드립니다.
